In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Orchestrating Multi-Agent Systems with Google Agent Development Kit

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/iamthuya/google-cloud-workshops/blob/main/ai-agents/agent-development-kit/orchestrating_multi_agent_systems.ipynb">
      <img src="https://avatars.githubusercontent.com/u/33467679?s=200&v=4" width="30px" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fiamthuya%2Fgoogle-cloud-workshops%2Fmain%2Fai-agents%2Fagent-development-kit%2Forchestrating_multi_agent_systems.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Run in Colab Enterprise
    </a>
  </td>      
  <td style="text-align: center">
    <a href="https://github.com/iamthuya/google-cloud-workshops/blob/main/ai-agents/agent-development-kit/orchestrating_multi_agent_systems.ipynb">
      <img src="https://github.blog/wp-content/uploads/2013/04/074d0b06-a5e3-11e2-8b7f-9f09eb2ddfae.jpg" width="55px" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/iamthuya/google-cloud-workshops/blob/main/ai-agents/agent-development-kit/orchestrating_multi-agent_systems.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>

| | |
|-|-|
|Author(s) | [Thu Ya Kyaw](https://github.com/iamthuya) |

## Environment Setup

### Install Necessary Libraries

Here you will install required Python packages for this lab.

In [10]:
%pip install -U -q google-adk

In [11]:
# Check ADK version
from google import adk

print(adk.__version__)

1.18.0


### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

Don't worry if you see a notification message like `Your session crashed for an unknown reason.` This is expected as you are shutting down the kernel from the same instance using code.


In [1]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Please wait until it is finished before continuing to the next step. ⚠️</b>
</div>

### Import Required Libraries

In [2]:
import os
import asyncio

from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, ToolContext
from google.genai import types

from IPython.display import Markdown, display

### Define Environment Variables

In [5]:
# Check if you are running this notebook from google colab
try:
  from google.colab import userdata
  os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
  os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
  print("*** Using Google AI Backend ***")

except:
  os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
  os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

  if os.getenv("GOOGLE_CLOUD_PROJECT") is None:  # check if project-id is already assigned
    project_id = "gdg-project-1-478304"  #@param {type:"string"}
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id

  import sys

  if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()


  print(("*** Using Vertex AI Backend ***\n"
         f"PROJECT: {os.environ['GOOGLE_CLOUD_PROJECT']}\n"
         f"LOCATION: {os.environ['GOOGLE_CLOUD_LOCATION']}"
       ))


*** Using Vertex AI Backend ***
PROJECT: gdg-project-1-478304
LOCATION: us-central1


## Sequential Agents

In this section, you'll create an agent that researches a topic, then writes in detail about it, and finally edits the output. These steps must happen in order: research first, then writing, and then editing, because you can't write without research, and you can't edit without a written piece.

In [13]:
# Define variables
GEMINI_MODEL_NAME = "gemini-2.5-flash"
APP_NAME = "sequential_app"
USER_ID = "sequential_user"
SESSION_ID = "sequential_session"

#### Initialize the agents and pipeline

In [14]:
# Define respective agents and pipeline
researcher = LlmAgent(
    name="ResearcherAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to use 'google_search' tool to find all the relavent information about a given topic and write a comprehensive report about that topic.",
    tools=[google_search],
    description="Researches about a topic",
    output_key="research_results"
)

writer = LlmAgent(
    name="WriterAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to write a comprehensive report using this information: {research_results}. Make sure to expend on all the topics from the provided information.",
    description="Write about a comprehensive report based on the provided information",
    output_key="comprehensive_report"
)

editor = LlmAgent(
    name="EditorAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to edit a report: {comprehensive_report}. Perform a quality check on the text and improve it if necessary.",
    description="Perform quality check on a written report",
)

sequential_pipeline = SequentialAgent(
    name="ContentPipelineAgent",
    sub_agents=[researcher, writer, editor],
    description="Executes a sequence of researching, writing, and editing.",
)

### Initialize Session & Runner

In [16]:
# Initialize Session
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Initialize Runner
sequential_runner = Runner(
    agent=sequential_pipeline,
    app_name=APP_NAME,
    session_service=session_service,
)

### Define a function to interact with Agent

In [20]:
# Define a function to interact with the agent
async def run_agent(runner, user_id, session_id, user_query):
    user_message = types.Content(
        role="user",
        parts=[types.Part(text=user_query)]
    )

    events = runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=user_message
    )

    output = {}
    async for event in events:
        if event.is_final_response():
            author = event.author
            report = event.content.parts[0].text
            output[author] = report

            display(Markdown(f"## {author}"))
            display(Markdown(f"{report}"))

    return output

In [19]:
topic = "Write about the current state of generative ai"
output = await run_agent(sequential_runner, USER_ID, SESSION_ID, topic)
report = output['EditorAgent']

## ResearcherAgent

The current state of generative artificial intelligence (AI) is characterized by rapid adoption, significant technological advancements, and widespread integration across various industries, while simultaneously navigating a complex landscape of challenges and ethical considerations. The technology has moved beyond novelty to become a core driver of innovation and efficiency.

### Rapid Adoption and Market Growth

Generative AI is experiencing an unprecedented surge in adoption, demonstrating a growth rate that surpasses the early stages of personal computers and the internet. As of August 2025, 54.6% of adults aged 18 to 64 are utilizing generative AI, representing a remarkable 10 percentage point increase within a mere 12 months. This widespread embrace is fueling robust market expansion; the global generative AI market is projected to reach US$699.50 billion by 2032 and expand at a compound annual growth rate (CAGR) of 33.0% from 2025 to 2032. Other projections show CAGRs ranging from 37.6% from 2025 to 2030 to 39.6% from 2024 to 2032. Additionally, some forecasts predict an annual growth rate of 46%.

Reflecting this trajectory, many organizations are substantially increasing their investments in generative AI. A significant number are already piloting or actively deploying generative AI in production environments, with 44% of organizations piloting programs and 10% having implemented them into production. Furthermore, 71% of organizations are regularly using generative AI in at least one business function. Worldwide generative AI spending is expected to total $644 billion in 2025, a 76.4% increase from 2024. Looking ahead, 92% of companies plan to invest in generative AI over the next three years.

### Key Trends and Advancements

The evolution of generative AI is being shaped by several transformative trends and continuous advancements:

*   **Smarter and Larger Models:** Ongoing research and development are consistently leading to the creation of more sophisticated, powerful, and capable AI models, with "smarter LLMs" emerging.
*   **Multimodal AI Capabilities:** Generative AI is moving beyond single data types to seamlessly integrate and process multiple forms of data, such as text, images, audio, and video. This leads to more intuitive and comprehensive applications. Leading models in 2025 include GPT-5, Claude 3, Google Gemini 2.0, Gemma 3, and LLaMA 3.2.
*   **Accessibility for Non-Technical Users:** The technology is becoming increasingly user-friendly, with accessible tools and user-friendly interfaces, including no-code/low-code platforms, enabling individuals without specialized technical expertise to leverage its powerful capabilities.
*   **Industry-Specific AI and Hyper-Personalization:** Tailored AI solutions are emerging for specific sectors, enabling highly personalized experiences and products for consumers and businesses alike.
*   **Autonomous AI Agents:** The development and anticipated rise of autonomous AI agents promise to handle increasingly complex tasks with minimal human intervention. It is forecast that 25% of enterprises using GenAI will deploy AI agents in 2025, growing to 50% by 2027.
*   **Enhanced Conversational AI:** Generative AI is significantly improving chatbots and virtual assistants, making their interactions more intelligent, natural, and human-like.
*   **Generative Design and Content Creation:** Creative fields are being revolutionized as generative AI assists in product design, the creation of diverse marketing content, and even the generation of music and videos.
*   **AI for App Development (No-Code/Low-Code):** Generative AI is empowering a broader range of users by enabling the creation of applications with minimal or no coding knowledge.
*   **Scientific Discovery:** The technology is playing an increasingly vital role in accelerating scientific research, including critical applications in drug discovery, materials science, and hypothesis generation.
*   **Automation of Complex Workflows:** Generative AI is extending its reach beyond simple automation to manage intricate business processes, such as supply chain management and the generation of legal contracts.

### Industry Applications

Generative AI is not merely influencing but actively disrupting and transforming numerous sectors:

*   **Healthcare:** Applications include accelerating drug discovery processes, generating synthetic clinical data for research and training, and personalizing patient engagement strategies.
*   **Manufacturing:** The technology is utilized for generative design, allowing for rapid iteration and optimization of product designs, and for predictive maintenance to minimize downtime.
*   **Finance:** Generative AI facilitates automated compliance procedures, generates comprehensive financial reports, and enables sophisticated risk scenario simulations.
*   **Retail & E-commerce:** It enhances customer experiences through AI-driven personalization, automated content generation for product descriptions and marketing, and intelligent conversational shopping assistants.
*   **Marketing and Customer Service:** Generative AI is used to create compelling content, improve customer support through smarter chatbots, and enable hyper-personalized marketing campaigns.
*   **Software Development:** Tools like GitHub Copilot are significantly boosting developer productivity by assisting with coding, debugging, and automating repetitive tasks. Software developers can experience 10-30% productivity gains when using Generative AI.

### Challenges and Limitations

Despite its rapid advancements, generative AI is contending with several significant challenges:

*   **Data Governance, Security, and Infrastructure Costs:** Managing the vast volumes of data required for training, ensuring its security against breaches, and covering the substantial computational and infrastructure costs remain major hurdles. For example, GenAI traffic surged 890% in 2024, leading to increased data loss prevention (DLP) incidents.
*   **Bias, Fairness, and Ethics:** A critical concern is the tendency of generative AI models to perpetuate biases present in their training data, potentially leading to unfair or discriminatory outputs. Ensuring transparency, explainability, and ethical deployment of these systems is paramount.
*   **Precision and Reliability:** Generated outputs can sometimes be erroneous, inconsistent, or outright fabricated ("hallucinations"). This highlights issues with the technology's inherent unpredictability and a current lack of complete understanding of its internal workings.
*   **Complex Integration:** Integrating generative AI tools into existing, often legacy, business systems can be a costly and complex undertaking, demanding specialized expertise and suitable technological infrastructure.
*   **Skills Gap and Workforce Adaptation:** A significant portion of the global workforce currently lacks the necessary skills to effectively utilize generative AI tools. This raises concerns about potential job displacement and the need for widespread upskilling.
*   **Intellectual Property (IP) and Licensing:** Prominent questions revolve around the protection of intellectual property for AI-generated assets and the complexities of licensing their use. There is also a risk of AI generating outputs that closely resemble existing copyrighted or trademarked third-party content.
*   **True Creativity vs. Remixing:** While generative AI excels at creating variations, synthesizing, and remixing existing ideas, the capacity for genuine creative leaps, profound conceptualization, and a deep understanding of subtle human emotions often remains a uniquely human domain.

### Future Outlook

Looking ahead, generative AI is poised for continued evolution and deeper integration across society and industry. The future trajectory will likely see a shift towards more mature business applications and the development of specialized, domain-specific solutions tailored to precise needs. A significant focus will be on the creation of advanced multimodal models capable of associating complex information across various modalities.

Simultaneously, there will be an increased emphasis on robust AI governance frameworks, comprehensive regulatory measures, and enhanced security protocols to mitigate risks and ensure the responsible development and deployment of this powerful technology. The overarching goal is to move beyond initial experimentation and pilot phases to fully scale AI's transformative impact across enterprises, fundamentally redefining operations, enhancing decision-making processes, and revolutionizing customer experiences.

## WriterAgent

## The Current State of Generative Artificial Intelligence: A Comprehensive Report

The current state of generative artificial intelligence (AI) is marked by profound dynamism, characterized by rapid adoption, significant technological advancements, and widespread integration across diverse industries. Generative AI has transcended its initial novelty, evolving into a fundamental driver of innovation and efficiency, yet it simultaneously navigates a complex landscape of challenges and ethical considerations.

### Rapid Adoption and Market Growth

Generative AI is experiencing an unprecedented surge in adoption, demonstrating a growth rate that surpasses the early stages of personal computers and the internet. As of August 2025, 54.6% of adults aged 18 to 64 are utilizing generative AI, representing a remarkable 10 percentage point increase within a mere 12 months. This widespread embrace is fueling robust market expansion; the global generative AI market is projected to reach US$699.50 billion by 2032 and expand at a compound annual growth rate (CAGR) of 33.0% from 2025 to 2032. Other projections show CAGRs ranging from 37.6% from 2025 to 2030 to 39.6% from 2024 to 2032. Additionally, some forecasts predict an annual growth rate of 46%.

Reflecting this trajectory, many organizations are substantially increasing their investments in generative AI. A significant number are already piloting or actively deploying generative AI in production environments, with 44% of organizations piloting programs and 10% having implemented them into production. Furthermore, 71% of organizations are regularly using generative AI in at least one business function. Worldwide generative AI spending is expected to total $644 billion in 2025, a 76.4% increase from 2024. Looking ahead, 92% of companies plan to invest in generative AI over the next three years.

### Key Trends and Advancements

The evolution of generative AI is being shaped by several transformative trends and continuous advancements:

*   **Smarter and Larger Models:** Ongoing research and development are consistently leading to the creation of more sophisticated, powerful, and capable AI models, with "smarter LLMs" emerging.
*   **Multimodal AI Capabilities:** Generative AI is moving beyond single data types to seamlessly integrate and process multiple forms of data, such as text, images, audio, and video. This leads to more intuitive and comprehensive applications. Leading models in 2025 include GPT-5, Claude 3, Google Gemini 2.0, Gemma 3, and LLaMA 3.2.
*   **Accessibility for Non-Technical Users:** The technology is becoming increasingly user-friendly, with accessible tools and user-friendly interfaces, including no-code/low-code platforms, enabling individuals without specialized technical expertise to leverage its powerful capabilities.
*   **Industry-Specific AI and Hyper-Personalization:** Tailored AI solutions are emerging for specific sectors, enabling highly personalized experiences and products for consumers and businesses alike.
*   **Autonomous AI Agents:** The development and anticipated rise of autonomous AI agents promise to handle increasingly complex tasks with minimal human intervention. It is forecast that 25% of enterprises using GenAI will deploy AI agents in 2025, growing to 50% by 2027.
*   **Enhanced Conversational AI:** Generative AI is significantly improving chatbots and virtual assistants, making their interactions more intelligent, natural, and human-like.
*   **Generative Design and Content Creation:** Creative fields are being revolutionized as generative AI assists in product design, the creation of diverse marketing content, and even the generation of music and videos.
*   **AI for App Development (No-Code/Low-Code):** Generative AI is empowering a broader range of users by enabling the creation of applications with minimal or no coding knowledge.
*   **Scientific Discovery:** The technology is playing an increasingly vital role in accelerating scientific research, including critical applications in drug discovery, materials science, and hypothesis generation.
*   **Automation of Complex Workflows:** Generative AI is extending its reach beyond simple automation to manage intricate business processes, such as supply chain management and the generation of legal contracts.

### Industry Applications

Generative AI is not merely influencing but actively disrupting and transforming numerous sectors:

*   **Healthcare:** Applications include accelerating drug discovery processes, generating synthetic clinical data for research and training, and personalizing patient engagement strategies.
*   **Manufacturing:** The technology is utilized for generative design, allowing for rapid iteration and optimization of product designs, and for predictive maintenance to minimize downtime.
*   **Finance:** Generative AI facilitates automated compliance procedures, generates comprehensive financial reports, and enables sophisticated risk scenario simulations.
*   **Retail & E-commerce:** It enhances customer experiences through AI-driven personalization, automated content generation for product descriptions and marketing, and intelligent conversational shopping assistants.
*   **Marketing and Customer Service:** Generative AI is used to create compelling content, improve customer support through smarter chatbots, and enable hyper-personalized marketing campaigns.
*   **Software Development:** Tools like GitHub Copilot are significantly boosting developer productivity by assisting with coding, debugging, and automating repetitive tasks. Software developers can experience 10-30% productivity gains when using Generative AI.

### Challenges and Limitations

Despite its rapid advancements, generative AI is contending with several significant challenges:

*   **Data Governance, Security, and Infrastructure Costs:** Managing the vast volumes of data required for training, ensuring its security against breaches, and covering the substantial computational and infrastructure costs remain major hurdles. For example, GenAI traffic surged 890% in 2024, leading to increased data loss prevention (DLP) incidents.
*   **Bias, Fairness, and Ethics:** A critical concern is the tendency of generative AI models to perpetuate biases present in their training data, potentially leading to unfair or discriminatory outputs. Ensuring transparency, explainability, and ethical deployment of these systems is paramount.
*   **Precision and Reliability:** Generated outputs can sometimes be erroneous, inconsistent, or outright fabricated ("hallucinations"). This highlights issues with the technology's inherent unpredictability and a current lack of complete understanding of its internal workings.
*   **Complex Integration:** Integrating generative AI tools into existing, often legacy, business systems can be a costly and complex undertaking, demanding specialized expertise and suitable technological infrastructure.
*   **Skills Gap and Workforce Adaptation:** A significant portion of the global workforce currently lacks the necessary skills to effectively utilize generative AI tools. This raises concerns about potential job displacement and the need for widespread upskilling.
*   **Intellectual Property (IP) and Licensing:** Prominent questions revolve around the protection of intellectual property for AI-generated assets and the complexities of licensing their use. There is also a risk of AI generating outputs that closely resemble existing copyrighted or trademarked third-party content.
*   **True Creativity vs. Remixing:** While generative AI excels at creating variations, synthesizing, and remixing existing ideas, the capacity for genuine creative leaps, profound conceptualization, and a deep understanding of subtle human emotions often remains a uniquely human domain.

### Future Outlook

Looking ahead, generative AI is poised for continued evolution and deeper integration across society and industry. The future trajectory will likely see a shift towards more mature business applications and the development of specialized, domain-specific solutions tailored to precise needs. A significant focus will be on the creation of advanced multimodal models capable of associating complex information across various modalities.

Simultaneously, there will be an increased emphasis on robust AI governance frameworks, comprehensive regulatory measures, and enhanced security protocols to mitigate risks and ensure the responsible development and deployment of this powerful technology. The overarching goal is to move beyond initial experimentation and pilot phases to fully scale AI's transformative impact across enterprises, fundamentally redefining operations, enhancing decision-making processes, and revolutionizing customer experiences.

## EditorAgent

This report is exceptionally well-written and comprehensive, demonstrating a high level of clarity, detail, and structure.

**Quality Check Results:**

1.  **Clarity and Conciseness:** The language is clear, direct, and professional. Technical terms are used appropriately or explained contextually (e.g., "hallucinations"). There is no unnecessary jargon or redundancy.
2.  **Accuracy:** Assuming the provided statistics and projections are accurate from the source data, they are integrated seamlessly and effectively bolster the claims. The ranges for CAGRs and specific dates add credibility.
3.  **Completeness:** The report covers all critical aspects of the current state of generative AI, including adoption, market growth, key technological trends, diverse industry applications, significant challenges, and a forward-looking outlook.
4.  **Flow and Structure:** The report is logically organized with clear headings and sub-sections. Transitions between paragraphs and sections are smooth, making it easy to follow the narrative and absorb the information. Bullet points are used effectively to present lists of trends, applications, and challenges, enhancing readability.
5.  **Grammar, Spelling, Punctuation:** No errors were found in grammar, spelling, or punctuation.
6.  **Professional Tone:** The tone is consistently authoritative, objective, and professional, suitable for a comprehensive report.
7.  **Impact and Engagement:** The report effectively communicates the rapid evolution and significant impact of generative AI. The use of specific statistics and real-world examples (e.g., GitHub Copilot, leading models) enhances engagement and understanding.

**Conclusion:**

The report is of outstanding quality and requires no improvements. It is a well-researched, meticulously organized, and clearly articulated document that thoroughly addresses the current state of generative artificial intelligence.

## Parallel Agents

Sometimes, running agents in parallel is desirable, particularly for tasks like translation. The Agent Developer Kit supports such parallel orchestration. In this session, you will create a parallel pipeline designed to translate a report into several languages simultaneously.

In [21]:
# Define variables
APP_NAME = "parallel_app"
USER_ID = "parallel_user"
SESSION_ID = "parallel_session"

#### Initialize the agents and pipeline

In [22]:
# Define respective agents and pipeline

kannada_translator = LlmAgent(
    name="KannadaAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to translate the given report into Kannada lanaguage",
    description="Translate a given report to Kannada",
    output_key="kannada_version"
)

thai_translator = LlmAgent(
    name="ThaiAgent",
    model=GEMINI_MODEL_NAME,
    instruction="Your task is to translate the given report into Thai lanaguage",
    description="Translate a given report to Thai",
    output_key="thai_version"
)

parallel_pipeline = ParallelAgent(
    name="TranslationPipelineAgent",
    sub_agents=[kannada_translator, thai_translator],
    description="Runs multiple tranlsation agents in parallel to translate the report.",
)

#### Initialize Session & Runner

In [25]:
# Initialize Session
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Initialize Runner
parallel_runner = Runner(
    agent=parallel_pipeline,
    app_name=APP_NAME,
    session_service=session_service,
)

In [24]:
prompt = f"translate this report: {report}"
output = await run_agent(parallel_runner, USER_ID, SESSION_ID, prompt)

## KannadaAgent

ಈ ವರದಿಯು ಅಸಾಧಾರಣವಾಗಿ ಉತ್ತಮವಾಗಿ ಬರೆಯಲ್ಪಟ್ಟಿದೆ ಮತ್ತು ಸಮಗ್ರವಾಗಿದೆ, ಇದು ಉನ್ನತ ಮಟ್ಟದ ಸ್ಪಷ್ಟತೆ, ವಿವರ ಮತ್ತು ರಚನೆಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.

**ಗುಣಮಟ್ಟ ಪರಿಶೀಲನೆ ಫಲಿತಾಂಶಗಳು:**

1.  **ಸ್ಪಷ್ಟತೆ ಮತ್ತು ಸಂಕ್ಷಿಪ್ತತೆ:** ಭಾಷೆ ಸ್ಪಷ್ಟವಾಗಿದೆ, ನೇರವಾಗಿದೆ ಮತ್ತು ವೃತ್ತಿಪರವಾಗಿದೆ. ತಾಂತ್ರಿಕ ಪದಗಳನ್ನು ಸೂಕ್ತವಾಗಿ ಬಳಸಲಾಗಿದೆ ಅಥವಾ ಸಂದರ್ಭಾನುಸಾರ ವಿವರಿಸಲಾಗಿದೆ (ಉದಾಹಹರಣೆಗೆ, "hallucinations"). ಅನಗತ್ಯ ಪರಿಭಾಷೆ ಅಥವಾ ಪುನರಾವರ್ತನೆ ಇಲ್ಲ.
2.  **ನಿಖರತೆ:** ಒದಗಿಸಿದ ಅಂಕಿಅಂಶಗಳು ಮತ್ತು ಮುನ್ನೋಟಗಳು ಮೂಲ ಡೇಟಾದಿಂದ ನಿಖರವಾಗಿವೆ ಎಂದು ಊಹಿಸಿ, ಅವುಗಳನ್ನು ಮನಬಂದಂತೆ ಸಂಯೋಜಿಸಲಾಗಿದೆ ಮತ್ತು ಹಕ್ಕುಗಳನ್ನು ಪರಿಣಾಮಕಾರಿಯಾಗಿ ಬಲಪಡಿಸುತ್ತವೆ. CAGR ಗಳು ಮತ್ತು ನಿರ್ದಿಷ್ಟ ದಿನಾಂಕಗಳ ಶ್ರೇಣಿಗಳು ವಿಶ್ವಾಸಾರ್ಹತೆಯನ್ನು ಹೆಚ್ಚಿಸುತ್ತವೆ.
3.  **ಸಂಪೂರ್ಣತೆ:** ವರದಿಯು ಉತ್ಪಾದಕ AI ನ ಪ್ರಸ್ತುತ ಸ್ಥಿತಿಯ ಎಲ್ಲಾ ನಿರ್ಣಾಯಕ ಅಂಶಗಳನ್ನು ಒಳಗೊಂಡಿದೆ, ಅವುಗಳೆಂದರೆ ಅಳವಡಿಕೆ, ಮಾರುಕಟ್ಟೆ ಬೆಳವಣಿಗೆ, ಪ್ರಮುಖ ತಾಂತ್ರಿಕ ಪ್ರವೃತ್ತಿಗಳು, ವಿವಿಧ ಉದ್ಯಮ ಅನ್ವಯಗಳು, ಗಮನಾರ್ಹ ಸವಾಲುಗಳು ಮತ್ತು ಭವಿಷ್ಯದ ದೃಷ್ಟಿಕೋನ.
4.  **ಹರಿವು ಮತ್ತು ರಚನೆ:** ವರದಿಯನ್ನು ಸ್ಪಷ್ಟ ಶೀರ್ಷಿಕೆಗಳು ಮತ್ತು ಉಪ-ವಿಭಾಗಗಳೊಂದಿಗೆ ತಾರ್ಕಿಕವಾಗಿ ಆಯೋಜಿಸಲಾಗಿದೆ. ಪ್ಯಾರಾಗಳು ಮತ್ತು ವಿಭಾಗಗಳ ನಡುವಿನ ಪರಿವರ್ತನೆಗಳು ಸುಗಮವಾಗಿವೆ, ಇದು ನಿರೂಪಣೆಯನ್ನು ಅನುಸರಿಸಲು ಮತ್ತು ಮಾಹಿತಿಯನ್ನು ಹೀರಿಕೊಳ್ಳಲು ಸುಲಭಗೊಳಿಸುತ್ತದೆ. ಪ್ರವೃತ್ತಿಗಳು, ಅನ್ವಯಗಳು ಮತ್ತು ಸವಾಲುಗಳ ಪಟ್ಟಿಗಳನ್ನು ಪ್ರಸ್ತುತಪಡಿಸಲು ಬುಲೆಟ್ ಪಾಯಿಂಟ್‌ಗಳನ್ನು ಪರಿಣಾಮಕಾರಿಯಾಗಿ ಬಳಸಲಾಗುತ್ತದೆ, ಇದು ಓದುವಿಕೆಯನ್ನು ಹೆಚ್ಚಿಸುತ್ತದೆ.
5.  **ವ್ಯಾಕರಣ, ಕಾಗುಣಿತ, ವಿರಾಮಚಿಹ್ನೆ:** ವ್ಯಾಕರಣ, ಕಾಗುಣಿತ ಅಥವಾ ವಿರಾಮಚಿಹ್ನೆಯಲ್ಲಿ ಯಾವುದೇ ದೋಷಗಳು ಕಂಡುಬಂದಿಲ್ಲ.
6.  **ವೃತ್ತಿಪರ ಧ್ವನಿ:** ಧ್ವನಿಯು ಸ್ಥಿರವಾಗಿ ಅಧಿಕೃತ, ವಸ್ತುನಿಷ್ಠ ಮತ್ತು ವೃತ್ತಿಪರವಾಗಿದೆ, ಇದು ಸಮಗ್ರ ವರದಿಗೆ ಸೂಕ್ತವಾಗಿದೆ.
7.  **ಪರಿಣಾಮ ಮತ್ತು ಆಕರ್ಷಣೆ:** ಉತ್ಪಾದಕ AI ಯ ವೇಗದ ವಿಕಸನ ಮತ್ತು ಗಮನಾರ್ಹ ಪರಿಣಾಮವನ್ನು ವರದಿಯು ಪರಿಣಾಮಕಾರಿಯಾಗಿ ಸಂವಹಿಸುತ್ತದೆ. ನಿರ್ದಿಷ್ಟ ಅಂಕಿಅಂಶಗಳು ಮತ್ತು ನೈಜ-ಪ್ರಪಂಚದ ಉದಾಹರಣೆಗಳ (ಉದಾಹರಣೆಗೆ, GitHub Copilot, ಪ್ರಮುಖ ಮಾದರಿಗಳು) ಬಳಕೆಯು ಆಕರ್ಷಣೆ ಮತ್ತು ತಿಳುವಳಿಕೆಯನ್ನು ಹೆಚ್ಚಿಸುತ್ತದೆ.

**ತೀರ್ಮಾನ:**

ವರದಿಯು ಅತ್ಯುತ್ತಮ ಗುಣಮಟ್ಟದ್ದಾಗಿದೆ ಮತ್ತು ಯಾವುದೇ ಸುಧಾರಣೆಗಳ ಅಗತ್ಯವಿಲ್ಲ. ಇದು ಉತ್ತಮವಾಗಿ ಸಂಶೋಧಿಸಲ್ಪಟ್ಟ, ನಿಖರವಾಗಿ ಆಯೋಜಿಸಲ್ಪಟ್ಟ ಮತ್ತು ಸ್ಪಷ್ಟವಾಗಿ ನಿರೂಪಿಸಲ್ಪಟ್ಟ ದಾಖಲೆಯಾಗಿದ್ದು, ಉತ್ಪಾದಕ ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆಯ ಪ್ರಸ್ತುತ ಸ್ಥಿತಿಯನ್ನು ಸಂಪೂರ್ಣವಾಗಿ ತಿಳಿಸುತ್ತದೆ.

## ThaiAgent

รายงานฉบับนี้เขียนได้ดีเยี่ยมและครอบคลุม แสดงให้เห็นถึงความชัดเจน รายละเอียด และโครงสร้างในระดับสูง

**ผลการตรวจสอบคุณภาพ:**

1.  **ความชัดเจนและกระชับ:** ภาษาที่ใช้มีความชัดเจน ตรงไปตรงมา และเป็นมืออาชีพ ศัพท์ทางเทคนิคถูกใช้ได้อย่างเหมาะสม หรือมีการอธิบายตามบริบท (เช่น "hallucinations") ไม่มีศัพท์แสงที่ไม่จำเป็นหรือความซ้ำซ้อน
2.  **ความถูกต้อง:** สมมติว่าสถิติและการคาดการณ์ที่ให้มามีความถูกต้องจากข้อมูลต้นฉบับ ข้อมูลเหล่านั้นถูกรวมเข้าด้วยกันอย่างราบรื่นและสนับสนุนข้อกล่าวอ้างได้อย่างมีประสิทธิภาพ ช่วงค่า CAGR และวันที่ระบุช่วยเพิ่มความน่าเชื่อถือ
3.  **ความสมบูรณ์:** รายงานครอบคลุมทุกแง่มุมที่สำคัญของสถานะปัจจุบันของ AI เชิงสร้างสรรค์ รวมถึงการนำไปใช้ การเติบโตของตลาด แนวโน้มเทคโนโลยีที่สำคัญ การประยุกต์ใช้ในอุตสาหกรรมที่หลากหลาย ความท้าทายที่สำคัญ และแนวโน้มในอนาคต
4.  **การจัดลำดับและการจัดโครงสร้าง:** รายงานมีการจัดระเบียบอย่างมีเหตุผลด้วยหัวข้อหลักและหัวข้อรองที่ชัดเจน การเปลี่ยนผ่านระหว่างย่อหน้าและส่วนต่างๆ เป็นไปอย่างราบรื่น ทำให้ง่ายต่อการติดตามเรื่องราวและทำความเข้าใจข้อมูล มีการใช้สัญลักษณ์แสดงหัวข้อย่อย (bullet points) อย่างมีประสิทธิภาพเพื่อนำเสนอรายการแนวโน้ม การประยุกต์ใช้ และความท้าทาย ซึ่งช่วยเพิ่มความสามารถในการอ่าน
5.  **ไวยากรณ์ การสะกดคำ และเครื่องหมายวรรคตอน:** ไม่พบข้อผิดพลาดด้านไวยากรณ์ การสะกดคำ หรือเครื่องหมายวรรคตอน
6.  **น้ำเสียงที่เป็นมืออาชีพ:** น้ำเสียงที่ใช้มีความน่าเชื่อถือ เป็นกลาง และเป็นมืออาชีพอย่างสม่ำเสมอ เหมาะสมกับรายงานที่ครอบคลุม
7.  **ผลกระทบและการดึงดูดความสนใจ:** รายงานสื่อสารให้เห็นถึงวิวัฒนาการที่รวดเร็วและผลกระทบที่สำคัญของ AI เชิงสร้างสรรค์ได้อย่างมีประสิทธิภาพ การใช้สถิติที่เฉพาะเจาะจงและตัวอย่างในโลกแห่งความเป็นจริง (เช่น GitHub Copilot, โมเดลชั้นนำ) ช่วยเพิ่มการมีส่วนร่วมและความเข้าใจ

**บทสรุป:**

รายงานฉบับนี้มีคุณภาพโดดเด่นและไม่จำเป็นต้องปรับปรุงแก้ไขใดๆ เป็นเอกสารที่ผ่านการวิจัยมาอย่างดี มีการจัดระเบียบอย่างละเอียดรอบคอบ และนำเสนอได้อย่างชัดเจน ซึ่งกล่าวถึงสถานะปัจจุบันของปัญญาประดิษฐ์เชิงสร้างสรรค์ได้อย่างสมบูรณ์

## Loop Agents

Some workflow involves repetition or iterative refinement, such as like revising code. ADK also support such workflows.



In [26]:
# Declare constants
APP_NAME = "loop_app" # New App Name
USER_ID = "loop_user"
SESSION_ID = "loop_session" # New Base Session ID
STATE_INITIAL_TOPIC = "Kopi Luwak"

# Define state variables
STATE_CURRENT_DOC = "current_document"
STATE_CRITICISM = "criticism"

# Define the exact phrase the Critic should use to signal completion
COMPLETION_PHRASE = "No major issues found."

#### Tools

In [27]:
# Define exit loop tool
def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the critique indicates no further changes are needed, signaling the iterative process should end."""
    print(f"[Tool Call] exit_loop triggered by {tool_context.agent_name}")
    tool_context.actions.escalate = True
    # Return empty dict as tools should typically return JSON-serializable output
    return {}

#### Initial Writer Agent

In [28]:
# STEP 1: Initial Writer Agent (Runs ONCE at the beginning)
initial_writer_agent = LlmAgent(
    name="InitialWriterAgent",
    model=GEMINI_MODEL_NAME,
    include_contents='none',
    instruction=f"""You are a Creative Writing Assistant tasked with starting a story.
    Write the *first draft* of a short story (aim for 2-4 sentences).
    Base the content *only* on the topic provided below. Try to introduce a specific element (like a character, a setting detail, or a starting action) to make it engaging.
    Topic: {{initial_topic}}

    Output *only* the story/document text. Do not add introductions or explanations.
""",
    description="Writes the initial document draft based on the topic, aiming for some initial substance.",
    output_key=STATE_CURRENT_DOC
)

#### Critic Agent

In [29]:
# STEP 2a: Critic Agent (Inside the Refinement Loop)
critic_agent_in_loop = LlmAgent(
    name="CriticAgent",
    model=GEMINI_MODEL_NAME,
    include_contents='none',
    # MODIFIED Instruction: More nuanced completion criteria, look for clear improvement paths.
    instruction=f"""You are a Constructive Critic AI reviewing a short document draft (typically 2-6 sentences). Your goal is balanced feedback.

    **Document to Review:**
    ```
    {{current_document}}
    ```

    **Task:**
    Review the document for clarity, engagement, and basic coherence according to the initial topic (if known).

    IF you identify 1-2 *clear and actionable* ways the document could be improved to better capture the topic or enhance reader engagement (e.g., "Needs a stronger opening sentence", "Clarify the character's goal"):
    Provide these specific suggestions concisely. Output *only* the critique text.

    ELSE IF the document is coherent, addresses the topic adequately for its length (should be 1 page long), and has no glaring errors or obvious omissions:
    Respond *exactly* with the phrase "{COMPLETION_PHRASE}" and nothing else. It doesn't need to be perfect, just functionally complete for this stage. Avoid suggesting purely subjective stylistic preferences if the core is sound.

    Do not add explanations. Output only the critique OR the exact completion phrase.
""",
    description="Reviews the current draft, providing critique if clear improvements are needed, otherwise signals completion.",
    output_key=STATE_CRITICISM
)

### Refine Agent

In [31]:
# STEP 2b: Refiner/Exiter Agent (Inside the Refinement Loop)
refiner_agent_in_loop = LlmAgent(
    name="RefinerAgent",
    model=GEMINI_MODEL_NAME,
    # Relies solely on state via placeholders
    include_contents='none',
    instruction=f"""You are a Creative Writing Assistant refining a document based on feedback OR exiting the process.
    **Current Document:**
    ```
    {{current_document}}
    ```
    **Critique/Suggestions:**
    {{criticism}}

    **Task:**
    Analyze the 'Critique/Suggestions'.
    IF the critique is *exactly* "{COMPLETION_PHRASE}":
    You MUST call the 'exit_loop' function. Do not output any text.
    ELSE (the critique contains actionable feedback):
    Carefully apply the suggestions to improve the 'Current Document'. Output *only* the refined document text.

    Do not add explanations. Either output the refined document OR call the exit_loop function.
""",
    description="Refines the document based on critique, or calls exit_loop if critique indicates completion.",
    tools=[exit_loop], # Provide the exit_loop tool
    output_key=STATE_CURRENT_DOC # Overwrites state['current_document'] with the refined version
)

#### Refinement Loop Agent

In [32]:
# STEP 2c: Refinement Loop Agent
refinement_loop = LoopAgent(
    name="RefinementLoop",
    # Agent order is crucial: Critique first, then Refine/Exit
    sub_agents=[
        critic_agent_in_loop,
        refiner_agent_in_loop,
    ],
    max_iterations=5 # Limit loops
)

#### Overall Sequential Agent

In [33]:
# STEP 3: Overall Sequential Pipeline
# For ADK tools compatibility, the root agent must be named `root_agent`
root_agent = SequentialAgent(
    name="IterativeWritingPipeline",
    sub_agents=[
        initial_writer_agent, # Run first to create initial doc
        refinement_loop       # Then run the critique/refine loop
    ],
    description="Writes an initial document and then iteratively refines it with critique using an exit tool."
)

In [34]:
# Initialize Session
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state={"initial_topic": "Briyani"}
)

# Initialize Runner
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

In [35]:
await run_agent(runner, USER_ID, SESSION_ID, "write a comprehensive essay")

## InitialWriterAgent

The aroma of slow-cooked basmati and spiced lamb, unmistakably briyani, snaked its way from the kitchen, teasing old Mr. Krishnan from his afternoon nap. His eyes, though heavy-lidded, lit up with the memory of his grandmother’s secret recipe, a warmth spreading through his chest even before the first bite. He could almost feel the tender meat dissolving on his tongue.

## CriticAgent

No major issues found.

[Tool Call] exit_loop triggered by RefinerAgent


AttributeError: 'NoneType' object has no attribute 'parts'

# Conclusions

You have explored how to create an agent using Gemini and Agent Development Kit.